# Training diagnostics: `train_base.py` metrics

Reads `logs/metrics.csv` (written by `train_base.py` when run with `--log-dir logs`)
and plots loss, grad norm, z-score outlier diagnostics, learning rate, throughput,
and MFU% as line charts over training steps.

Run training first, e.g.:

```
python train_base.py --model d12 --max-steps 500 --log-dir logs
```

then re-run this notebook.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

METRICS_PATH = os.path.join("logs", "metrics.csv")
DARK_MODE = False  # flip to True for the dark-surface variant

In [ ]:
# Chart palette (light/dark), following a fixed categorical hue order so
# identity never depends on color alone -- see the legend on multi-series panels.

THEME = {
    "light": {
        "surface": "#fcfcfb",
        "text_primary": "#0b0b0b",
        "text_secondary": "#52514e",
        "text_muted": "#898781",
        "grid": "#e1e0d9",
        "axis": "#c3c2b7",
        "series": ["#2a78d6", "#1baf7a", "#eda100", "#4a3aa7", "#e34948"],
    },
    "dark": {
        "surface": "#1a1a19",
        "text_primary": "#ffffff",
        "text_secondary": "#c3c2b7",
        "text_muted": "#898781",
        "grid": "#2c2c2a",
        "axis": "#383835",
        "series": ["#3987e5", "#199e70", "#c98500", "#9085e9", "#e66767"],
    },
}


def get_theme() -> dict:
    return THEME["dark" if DARK_MODE else "light"]


def new_figure(ncols: int = 1, nrows: int = 1, figsize=None):
    theme = get_theme()
    if figsize is None:
        figsize = (6.5 * ncols, 4.0 * nrows)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, facecolor=theme["surface"])
    for ax in np.atleast_1d(axes).ravel():
        style_axis(ax, theme)
    return fig, axes


def style_axis(ax, theme: dict) -> None:
    ax.set_facecolor(theme["surface"])
    for spine_name, spine in ax.spines.items():
        if spine_name in ("top", "right"):
            spine.set_visible(False)
        else:
            spine.set_color(theme["axis"])
            spine.set_linewidth(0.8)
    ax.grid(True, linewidth=0.8, color=theme["grid"], solid_capstyle="butt")
    ax.set_axisbelow(True)
    ax.tick_params(colors=theme["text_muted"], labelsize=9)
    ax.xaxis.label.set_color(theme["text_secondary"])
    ax.yaxis.label.set_color(theme["text_secondary"])
    ax.title.set_color(theme["text_primary"])


def end_label(ax, x, y, text: str, color: str, theme: dict) -> None:
    """Marks the last point of a line and labels its value, instead of labeling every point."""
    if len(x) == 0:
        return
    ax.plot(x[-1], y[-1], marker="o", markersize=8, color=color,
             markeredgecolor=theme["surface"], markeredgewidth=2, zorder=5)
    ax.annotate(text, xy=(x[-1], y[-1]), xytext=(8, 0), textcoords="offset points",
                va="center", fontsize=9, color=theme["text_primary"])

In [ ]:
if not os.path.exists(METRICS_PATH):
    raise FileNotFoundError(
        f"{METRICS_PATH} not found. Run train_base.py with --log-dir logs first, "
        f"e.g.: python train_base.py --model d12 --max-steps 500 --log-dir logs"
    )

df = pd.read_csv(METRICS_PATH)
df.tail()

## Loss

In [ ]:
theme = get_theme()
fig, ax = new_figure()

ax.plot(df["step"], df["train_loss"], linewidth=2, color=theme["series"][0], label="train loss")
ax.plot(df["step"], df["val_loss"], linewidth=2, color=theme["series"][1], label="val loss")
end_label(ax, df["step"].to_numpy(), df["train_loss"].to_numpy(),
          f"{df['train_loss'].iloc[-1]:.3f}", theme["series"][0], theme)
end_label(ax, df["step"].to_numpy(), df["val_loss"].to_numpy(),
          f"{df['val_loss'].iloc[-1]:.3f}", theme["series"][1], theme)

ax.set_title("Training and validation loss")
ax.set_xlabel("step")
ax.set_ylabel("cross-entropy loss")
legend = ax.legend(frameon=False, labelcolor=theme["text_secondary"])
plt.show()

## Gradient norm and z-score outlier diagnostics

The z-score panels start as gaps (`NaN`) until the `OutlierDetector`'s window
(128 steps by default) fills up, matching the `(+nanz)` seen in the console log
during early training. The dashed lines mark a +/-3 sigma outlier threshold.

In [ ]:
theme = get_theme()
fig, (ax_norm, ax_z) = new_figure(ncols=2)

ax_norm.plot(df["step"], df["grad_norm"], linewidth=2, color=theme["series"][2])
end_label(ax_norm, df["step"].to_numpy(), df["grad_norm"].to_numpy(),
          f"{df['grad_norm'].iloc[-1]:.3f}", theme["series"][2], theme)
ax_norm.set_title("Gradient norm")
ax_norm.set_xlabel("step")
ax_norm.set_ylabel("grad norm")

ax_z.plot(df["step"], df["loss_zscore"], linewidth=2, color=theme["series"][0], label="loss z-score")
ax_z.plot(df["step"], df["grad_norm_zscore"], linewidth=2, color=theme["series"][3], label="grad norm z-score")
ax_z.axhline(3, linestyle="--", linewidth=1, color=theme["text_muted"])
ax_z.axhline(-3, linestyle="--", linewidth=1, color=theme["text_muted"])
ax_z.set_title("Outlier z-scores")
ax_z.set_xlabel("step")
ax_z.set_ylabel("z-score")
ax_z.legend(frameon=False, labelcolor=theme["text_secondary"])
plt.show()

## Learning rate schedule

In [ ]:
theme = get_theme()
fig, ax = new_figure()

ax.plot(df["step"], df["learning_rate"], linewidth=2, color=theme["series"][0])
end_label(ax, df["step"].to_numpy(), df["learning_rate"].to_numpy(),
          f"{df['learning_rate'].iloc[-1]:.2e}", theme["series"][0], theme)
ax.set_title("Learning rate")
ax.set_xlabel("step")
ax.set_ylabel("learning rate")
plt.show()

## Throughput and MFU%

MFU% is only computed for GPUs in `train_base.py`'s hardcoded peak-FLOPS table
(H100/A100/V100/L40/L4/RTX 4090/RTX 3090/T4). On an unrecognized GPU the column
is empty and the panel below shows a fallback message instead of an empty plot.

In [ ]:
theme = get_theme()
fig, (ax_tok, ax_mfu) = new_figure(ncols=2)

ax_tok.plot(df["step"], df["tokens_per_second"], linewidth=2, color=theme["series"][1])
end_label(ax_tok, df["step"].to_numpy(), df["tokens_per_second"].to_numpy(),
          f"{df['tokens_per_second'].iloc[-1]:.0f}", theme["series"][1], theme)
ax_tok.set_title("Throughput")
ax_tok.set_xlabel("step")
ax_tok.set_ylabel("tokens / sec")

mfu_pct = df["mfu"] * 100.0
if mfu_pct.notna().any():
    ax_mfu.plot(df["step"], mfu_pct, linewidth=2, color=theme["series"][4])
    valid = mfu_pct.dropna()
    end_label(ax_mfu, df.loc[valid.index, "step"].to_numpy(), valid.to_numpy(),
               f"{valid.iloc[-1]:.1f}%", theme["series"][4], theme)
    ax_mfu.set_ylabel("MFU %")
else:
    ax_mfu.text(0.5, 0.5, "MFU not available\n(GPU not in peak-FLOPS table)",
                ha="center", va="center", color=theme["text_muted"], transform=ax_mfu.transAxes)
    ax_mfu.set_yticks([])
ax_mfu.set_title("Matrix FLOPS utilization")
ax_mfu.set_xlabel("step")
plt.show()

## Summary table

In [ ]:
summary = pd.DataFrame({
    "final train loss": [df["train_loss"].iloc[-1]],
    "final val loss": [df["val_loss"].iloc[-1]],
    "best val loss": [df["val_loss"].min()],
    "final grad norm": [df["grad_norm"].iloc[-1]],
    "avg tokens/sec": [df["tokens_per_second"].mean()],
    "avg MFU %": [df["mfu"].mean() * 100.0 if df["mfu"].notna().any() else float("nan")],
    "total steps": [int(df["step"].iloc[-1])],
})
summary